# 1일차 실습 P2-1 — 학습 루프 채우기

- 수업 중 이 실습 슬라이드가 나오면 풂 · **맨 위 준비 셀부터** 위에서 아래로 실행
- 셀 실행: 셀을 누르고 **Shift + Enter** 또는 셀 왼쪽 ▶
- Colab 에서 처음 열 때 경고 창이 뜨면 '계속' · 새 파일은 준비 셀에서 데이터를 받느라 몇십 초 걸릴 수 있음
- 빈칸은 `____` · 빈칸을 모두 바꾼 뒤 **그 셀부터 다시 실행**
- 막히면 셀 이름과 **에러 칸 맨 아래 줄**을 채팅으로
- P2-1 을 끝냈거나 안내가 나오면(못 끝냈어도) `PRACTICE_P2_BUG` 파일을 열어 P2-2 를 풂

## 에러를 읽는 법

- 에러 칸 첫 줄(`...Error    Traceback ...`)은 제목일 뿐 · 무엇이 틀렸는지는 **맨 아래 줄**
- 중간의 `---->` 화살표 줄 · torch 안쪽 칸은 처음엔 건너뜀 · **위에 찍힌 `[안내]` 문장과 맨 아래 줄부터** 읽음

| 맨 아래 줄에 보이는 말 | 뜻 | 먼저 볼 곳 |
|---|---|---|
| `name '____' is not defined` | 빈칸이 남음 | 화살표가 가리키는 줄 |
| `name 'nn'` · `'Xtr'` · `'MyAE'` · `'MyConvAE'` · `'크기_검사'` is not defined | 그 이름을 만든 셀을 안 돌렸거나 런타임이 끊김 | 준비 셀부터 순서대로 다시 |
| `name 'latent_dimm'` · `'Sigmoid'` · `'optimizer'` · `'ae'` is not defined | 내가 친 이름이 틀림 · 철자 · `nn.` 빠짐 · 학습 함수 안 이름은 `model` · `opt` | 화살표가 가리키는 줄 |
| `unexpected indent` · `unindent` | 줄 앞 칸 수가 위아래 줄과 다름 | 바로 위 줄과 같은 칸에서 시작하게 |
| `Perhaps you forgot a comma?` · `'(' was never closed` · `unmatched ')'` | 쉼표나 괄호 | `^` 표시 자리 |
| `mat1 and mat2 shapes cannot be multiplied (AxB and CxD)` | Linear 에 들어온 값 개수 B 와 Linear 첫 숫자 C 가 다름 | `Flatten` 과 `Linear` 첫 숫자 |
| `is not a Module subclass` | 부품 뒤 `()` 가 빠짐 | `nn.ReLU` → `nn.ReLU()` |
| `expected input[...] to have a channels, but got c channels` | 층의 첫 숫자(받는 채널)가 들어온 채널과 다름 | 마지막으로 찍힌 층의 **다음** 층 첫 숫자 |
| `The size of tensor a (..) must match the size of tensor b (..)` | 출력 크기가 정답 크기와 다름 | 층마다 찍힌 모양 · 이름의 대소문자(`x` · `X`) |

- **에러가 없는데** loss 가 0.2 근처에서 안 내려가고 그림이 회색 네모 → 학습이 안 됐을 수 있음 · `backward()` · `step()` 괄호부터 확인

## 준비

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets

tr = datasets.MNIST("data", train=True, download=True)
te = datasets.MNIST("data", train=False, download=True)
Xtr = tr.data.float().div(255).unsqueeze(1)   # 학습 60,000장 · 값 0~1
Xte = te.data.float().div(255).unsqueeze(1)   # 평가 10,000장
SHOW = [5459, 9898, 926, 1150, 5691, 2400, 7792, 950]   # 평가 이미지 중 숫자 0~7 한 장씩


@torch.no_grad()
def run(model, X):                      # 2,000장씩 넣어 복원을 모음
    model.eval()
    return torch.cat([model(X[i:i + 2000]) for i in range(0, len(X), 2000)])


def show(rows):                         # 같은 여덟 장을 줄마다 나란히
    fig, axes = plt.subplots(len(rows), 8, figsize=(8, 1.15 * len(rows)))
    for r, (name, X) in enumerate(rows):
        for c in range(8):
            axes[r, c].imshow(X[SHOW[c], 0], cmap="gray", vmin=0, vmax=1)
            axes[r, c].axis("off")
        axes[r, 0].set_title(name, loc="left", fontsize=10)
    plt.tight_layout()
    plt.show()


print("준비 끝 · 학습", tuple(Xtr.shape), " 평가", tuple(Xte.shape))

## P2-1. 학습 루프 채우기

1. 위 **준비** 셀 → 아래 **모델** 셀을 실행(모델은 준비된 것 · 오전 AutoEncoder 와 같은 모양 · 잠재 16)
2. 학습 루프 빈칸 세 곳을 채움: 손실의 두 자리 · 기울기 계산 · 가중치 갱신
   - 이 함수 안에서 모델 이름은 **`model`** · 옵티마이저는 **`opt`** (오전 강사 코드의 `ae` 자리)
   - `backward` · `step` 은 괄호 `()` 까지 · 빠뜨리면 에러 없이 학습이 안 됨
3. 실행하면 학습 이미지 10,000장으로 3 에폭 학습하고 원본과 나란히 보여 줌
   - 정상이면 3 에폭 동안 loss 가 대략 0.06 → 0.04 로 내려가고(강사 컴퓨터 기준 · 조금 달라도 됨) 그림 칸마다 넣은 숫자와 비슷한 모양 · loss 만 내려가서는 정상이라고 할 수 없음 · 3 에폭이라 그림은 흐림
- 막히면 코드 셀 아래 **힌트 1** → **힌트 2** 순서로 하나씩 펼침
- P1 에서 만든 모델을 쓰고 싶으면 모델 셀 대신 P1 파일의 `LATENT` · `MyAE` 셀을 붙여 넣어 실행해도 됨

In [ ]:
# 모델 — 오전에 본 AutoEncoder 와 같은 모양(잠재 16) · 이 셀은 고치지 않고 실행
LATENT = 16


class MyAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU(), nn.Linear(128, 784), nn.Sigmoid())

    def forward(self, x):
        return self.decoder(self.encoder(x)).reshape(-1, 1, 28, 28)


print("모델 준비 · LATENT =", LATENT)

In [ ]:
def train(model, X, epochs=3):
    torch.manual_seed(0)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X))            # 번호를 섞음
        for i in range(0, len(X), 128):          # 앞에서부터 128장씩
            x = X[perm[i:i + 128]]
            opt.zero_grad()                      # 앞 묶음에서 쌓인 기울기를 비움
            loss = F.mse_loss(____, ____)        # 첫 자리: 모델이 내놓은 복원 · 둘째 자리: 맞혀야 할 정답
            ____                                 # 기울기 계산 · 괄호 () 까지
            ____                                 # 가중치 갱신 · 괄호 () 까지
        print(f"epoch {epoch + 1}  loss {loss.item():.4f}")
    if loss.item() > 0.15 or loss.item() < 1e-6:
        print("[확인] loss 가 0.2 근처에 멈추거나 0 이면 학습이 안 된 것일 수 있음 → backward() · step() 괄호 · 손실의 두 자리 · 두 줄이 opt.zero_grad() 와 같은 칸에서 시작하는지 · 막히면 셀 아래 힌트 1")
    return model


torch.manual_seed(0)
p2 = train(MyAE(LATENT), Xtr[:10000])
show([("original", Xte), (f"latent {LATENT} · 3 epochs", run(p2, Xte))])

<details><summary><b>힌트 1</b> — 막힐 때만 펼침</summary>

- 오전 '2. 학습' 셀에서 강사가 친 줄들과 같은 모양 · 그 셀의 `ae` 자리가 이 함수에서는 `model`

</details>

<details><summary><b>힌트 2</b> — 힌트 1 로도 막힐 때</summary>

- 손실의 첫 자리: 이번 묶음 `x` 를 모델에 넣어 나온 것 · 둘째 자리: 오늘 모델이 맞혀야 하는 것(오늘은 넣은 이미지를 그대로 되돌리기)
- 기울기 계산은 `loss` 에서 부르고, 가중치 갱신은 `opt` 에서 부름 · 둘 다 괄호 `()` 까지

</details>